# GeMoSC Classifier Re-run (with reference IDs, saved splits, and saved predictions)

This notebook validates the GeMoSC annotation by fine-tuning DistilBERT to recover
each label from dialogue alone. It is the reproducibility version: it carries a
stable reference identifier (`ref_id`) through every step, saves the exact
train/test split for each task, and saves each test character's true and predicted
label together with its dialogue, so that concrete success/failure examples can be
quoted in the thesis.

**Before running:** set the runtime to **T4 GPU** (Runtime -> Change runtime type -> T4 GPU),
and upload `classifier_input.csv` (produced locally by `make_classifier_input.py`).


In [ ]:
!pip -q install transformers==4.44.2 torch scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import numpy as np, pandas as pd, torch
from sklearn.model_selection import GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
import matplotlib.pyplot as plt, seaborn as sns

SEED = 42
MODEL = "distilbert-base-uncased"
MAXLEN = 256
EPOCHS = 4
LR = 2e-5
np.random.seed(SEED); torch.manual_seed(SEED)
print("CUDA available:", torch.cuda.is_available())

In [ ]:
df = pd.read_csv("classifier_input.csv", dtype=str)
df["clean_dialogue"] = df["clean_dialogue"].fillna("")
# film id as grouping key
df["film_id"] = df["IMDb ID"].astype(str).str.replace(r"\D","",regex=True).str.zfill(7)
# ref_id and genre_macro already come from the canonical file; no renumbering here.
assert "ref_id" in df.columns and "genre_macro" in df.columns
print("rows:", len(df), " films:", df["film_id"].nunique(),
      " unique ref_ids:", df["ref_id"].nunique())
df.head(3)

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL)

class DS(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.enc = tok(list(texts), truncation=True, padding=True, max_length=MAXLEN)
        self.labels = list(labels)
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        item = {k: torch.tensor(v[i]) for k,v in self.enc.items()}
        item["labels"] = torch.tensor(self.labels[i]); return item

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kw):
        super().__init__(**kw)
        self.class_weights = class_weights
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out = model(**inputs); logits = out.logits
        w = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss = torch.nn.functional.cross_entropy(logits, labels, weight=w)
        return (loss, out) if return_outputs else loss

In [ ]:
def run_task(name, sub, label_col, label_map=None):
    sub = sub.copy()
    if label_map: sub["y"] = sub[label_col].map(label_map)
    else: sub["y"] = sub[label_col]
    sub = sub[sub["y"].notna()].copy()
    classes = sorted(sub["y"].unique())
    c2i = {c:i for i,c in enumerate(classes)}
    sub["yi"] = sub["y"].map(c2i)

    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    tr, te = next(gss.split(sub, groups=sub["film_id"]))
    train, test = sub.iloc[tr].copy(), sub.iloc[te].copy()
    assert set(train["film_id"]) & set(test["film_id"]) == set(), "film leak!"

    # save the exact split by ref_id
    train[["ref_id","IMDb ID","title","character","y"]].to_csv(f"{name}_train_ids.csv", index=False)
    test [["ref_id","IMDb ID","title","character","y"]].to_csv(f"{name}_test_ids.csv",  index=False)

    # robust class weights: compute over classes present in TRAIN, fill 1.0 for any absent
    present = np.unique(train["yi"].values)
    w = np.ones(len(classes), dtype=np.float64)
    cw_present = compute_class_weight("balanced", classes=present, y=train["yi"].values)
    for ci, wv in zip(present, cw_present):
        w[int(ci)] = wv
    cw = torch.tensor(w, dtype=torch.float)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=len(classes))
    args = TrainingArguments(output_dir=f"out_{name}", num_train_epochs=EPOCHS,
        per_device_train_batch_size=16, per_device_eval_batch_size=32,
        learning_rate=LR, logging_steps=50, report_to="none", seed=SEED,
        save_strategy="no")
    trainer = WeightedTrainer(class_weights=cw, model=model, args=args,
        train_dataset=DS(train["clean_dialogue"], train["yi"]))
    trainer.train()

    pred = trainer.predict(DS(test["clean_dialogue"], test["yi"]))
    yhat = pred.predictions.argmax(-1)
    i2c = {i:c for c,i in c2i.items()}
    test["pred"] = [i2c[i] for i in yhat]

    # majority baseline
    maj = train["y"].value_counts().idxmax()
    base_acc = (test["y"]==maj).mean()
    base_f1  = f1_score(test["yi"], [c2i[maj]]*len(test), average="macro")
    acc = accuracy_score(test["yi"], yhat)
    mf1 = f1_score(test["yi"], yhat, average="macro")
    print(f"\n=== {name} ===  train {len(train)} / test {len(test)}")
    print(f"baseline  acc {base_acc:.3f}  macroF1 {base_f1:.3f}")
    print(f"model     acc {acc:.3f}  macroF1 {mf1:.3f}")

    # save per-character predictions WITH dialogue for the thesis examples
    test[["ref_id","IMDb ID","title","character","y","pred","clean_dialogue"]]\
        .rename(columns={"y":"true"}).to_csv(f"{name}_predictions.csv", index=False)

    # confusion matrix figure
    cm = confusion_matrix(test["y"], test["pred"], labels=classes)
    plt.figure(figsize=(1.2*len(classes)+2, 1.0*len(classes)+2))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=classes, yticklabels=classes)
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(name)
    plt.tight_layout(); plt.savefig(f"cm_{name}.png", dpi=150); plt.show()

    return dict(task=name, base_acc=base_acc, model_acc=acc,
                base_f1=base_f1, model_f1=mf1)

In [ ]:
results = []
results.append(run_task("gender", df[df["gender_norm"].isin(["male","female"])], "gender_norm"))
GENRES7 = ["Comedy","Action/Adventure","Drama","Biography","Crime","Horror","SciFi/Fantasy"]
results.append(run_task("genre",  df[df["genre_macro"].isin(GENRES7)], "genre_macro"))

gen5 = ["Silent","Boomer","Gen X","Millennial","Gen Z"]
results.append(run_task("generation5", df[df["generation"].isin(gen5)], "generation"))

m3 = {"Silent":"Older","Boomer":"Older","Gen X":"Middle","Millennial":"Younger","Gen Z":"Younger"}
results.append(run_task("generation3", df[df["generation"].isin(gen5)], "generation", m3))

pd.DataFrame(results).to_csv("classifier_results.csv", index=False)
pd.DataFrame(results)

## Outputs produced

For each task (`gender`, `genre`, `generation5`, `generation3`):
- `<task>_train_ids.csv`, `<task>_test_ids.csv` — the exact split, by `ref_id`
- `<task>_predictions.csv` — every test character's `ref_id`, true label, predicted label, and dialogue
- `cm_<task>.png` — confusion matrix

Plus `classifier_results.csv` (the summary table).

**Download all of these** (Files panel, left) and send them back, so the thesis can quote
real success/failure examples and record the exact splits.
